In [ ]:
!pip install scapy

In [4]:
# Примечание: для реального перехвата (sniff) нужны права администратора
# В Jupyter notebook это обычно не работает
# Используйте scapy в обычном Python-скрипте с sudo

from scapy.all import sniff

# Перехват пакетов
packets = sniff(count=100)  # Перехватить 10 пакетов
packets.summary()  # Краткая информация

# Детальная информация о пакете
packet = packets[0]
packet.show()  # Полная структура пакета

Ether / IP / TCP 192.168.0.101:60239 > 209.85.233.138:https PA / Raw
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:56230 PA / Raw
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:64527 PA / Raw
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:56231 PA / Raw
Ether / IP / TCP 192.168.0.101:61929 > 208.115.231.38:6568 A / Raw
Ether / IP / TCP 208.115.231.38:6568 > 192.168.0.101:61929 A
Ether / IP / TCP 192.168.0.101:56231 > 149.154.167.99:https A
Ether / IP / TCP 192.168.0.101:56230 > 149.154.167.99:https A
Ether / IP / TCP 192.168.0.101:64527 > 149.154.167.99:https A
Ether / IP / UDP 192.168.0.1:33730 > 255.255.255.255:7437 / Raw
Ether / IP / TCP 192.168.0.101:60229 > 209.85.233.101:https A / Raw
Ether / IP / TCP 192.168.0.101:60239 > 209.85.233.138:https PA / Raw
Ether / IP / TCP 192.168.0.101:56229 > 149.154.167.99:https PA / Raw
Ether / IP / TCP 192.168.0.101:60236 > 64.233.161.138:https PA / Raw
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:56229 A / Padding

In [6]:
from scapy.all import sniff
from scapy.layers.http import HTTPRequest, HTTPResponse


def process_packet(packet):
    """Обработка HTTP пакетов"""
    if packet.haslayer(HTTPRequest):
        http_layer = packet[HTTPRequest]
        print(f"[HTTP REQUEST] {http_layer.Method} {http_layer.Host}{http_layer.Path}")

        # Извлечение заголовков
        if http_layer.Authorization:
            print(f"Обнаружена авторизация: {http_layer.Authorization[:50]}...")

        # Дополнительная информация
        if http_layer.User_Agent:
            print(f"User-Agent: {http_layer.User_Agent[:80]}")

    if packet.haslayer(HTTPResponse):
        http_layer = packet[HTTPResponse]
        print(f"[HTTP RESPONSE] Status: {http_layer.Status_Code}")

        # Размер ответа
        if hasattr(http_layer, 'Content_Length'):
            print(f"Content-Length: {http_layer.Content_Length}")

try:
    #packets = sniff(filter="tcp port 80", prn=process_packet, timeout=20)
    packets = sniff(filter="tcp port 443", prn=process_packet, timeout=50)

    print(f"Перехвачено пакетов: {len(packets)}")

    http_requests = [p for p in packets if p.haslayer(HTTPRequest)]
    http_responses = [p for p in packets if p.haslayer(HTTPResponse)]

    print(f"HTTP запросов: {len(http_requests)}")
    print(f"HTTP ответов: {len(http_responses)}")

    if len(packets) == 0:
        print("Трафик не обнаружен.")

except KeyboardInterrupt:
    print("Перехват прерван пользователем")
except Exception as e:
    print(f"Ошибка: {e}")

Перехвачено пакетов: 2104
HTTP запросов: 0
HTTP ответов: 0


In [7]:
print(packets)

<Sniffed: TCP:2104 UDP:0 ICMP:0 Other:0>


In [8]:
print(packets[0].summary())

Ether / IP / TCP 192.168.0.101:56230 > 149.154.167.99:https PA / Raw


In [9]:
pkt = packets[0]
pkt.show()

###[ Ethernet ]###
  dst       = 40:3f:8c:e0:94:5e
  src       = 00:d8:61:a5:da:5e
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 189
     id        = 3938
     flags     = DF
     frag      = 0
     ttl       = 128
     proto     = tcp
     chksum    = 0x0
     src       = 192.168.0.101
     dst       = 149.154.167.99
     \options   \
###[ TCP ]###
        sport     = 56230
        dport     = https
        seq       = 2898873980
        ack       = 2293724536
        dataofs   = 5
        reserved  = 0
        flags     = PA
        window    = 64141
        chksum    = 0xfeba
        urgptr    = 0
        options   = []
###[ Raw ]###
           load      = b'\x17\x03\x03\x00\x90zz\xcc\x92\x05\x95R\x81&\xc4,\x8dK\xaf\xcd\xcf\xa2\xa9\x00\xdc\xc1/i\x90e\xb7f\xa9S*\x00\xee\x95*\xbc\xd9\xe0q\xc1@Z|6BRpw\xfa\x95\xdf\n\xb2[Z\x94\x00\xbc\xb9\x19\x03\xcfC\x134hj\xae\x16\xfd\x1a\x03\x9cw\xf1\xb3\xae\xae\xfc\xf94 \xa3l\xc9)\x1d\x08\

In [10]:
if pkt.haslayer('IP'):
    src_ip = pkt['IP'].src
    dst_ip = pkt['IP'].dst
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")


Откуда: 192.168.0.101 -> Куда: 149.154.167.99


In [11]:
if pkt.haslayer('Raw'):
    payload = pkt['Raw'].load
    print(f"Данные в байтах: {payload}")
    # Попытка декодировать в текст (если это не шифрованный трафик)
    try:
        print(f"Текст: {payload.decode('utf-8')}")
    except:
        print("Данные не являются текстом (бинарные или зашифрованы)")


Данные в байтах: b'\x17\x03\x03\x00\x90zz\xcc\x92\x05\x95R\x81&\xc4,\x8dK\xaf\xcd\xcf\xa2\xa9\x00\xdc\xc1/i\x90e\xb7f\xa9S*\x00\xee\x95*\xbc\xd9\xe0q\xc1@Z|6BRpw\xfa\x95\xdf\n\xb2[Z\x94\x00\xbc\xb9\x19\x03\xcfC\x134hj\xae\x16\xfd\x1a\x03\x9cw\xf1\xb3\xae\xae\xfc\xf94 \xa3l\xc9)\x1d\x08\xb9\x1c~G\xdd9\xe1\xbbe\xa0\xc1\xf8\xa7\xd6\xb82\tn8p\x17\x8d\xb2\x99Gi\x1c\x9e\xd8zA\x16\x13\xe3\x1e\xeb\xe2\xa5ly+\x88\xc6\xaa\xce-&\xf4\xe6\xfb \xbb\x1c\xe0\xd5T\xce'
Данные не являются текстом (бинарные или зашифрованы)


In [20]:
from scapy.all import sniff, TCP, Raw

def process_http(packet):
    # Проверяем, есть ли в пакете TCP и данные (Raw)
    if packet.haslayer(Raw):
        payload = packet[Raw].load
        try:
            # Декодируем байты в текст
            data = payload.decode('utf-8', errors='ignore')
            
            # Ищем характерные признаки HTTP (GET, POST, HTTP/1.1)
            if "HTTP" in data:
                print(f"\n[+] Перехвачен HTTP пакет от {packet[0][1].src}:")
                print("-" * 40)
                print(data)
                print("-" * 40)
        except Exception:
            pass

# Запускаем прослушивание только порта 80 (HTTP)
print("Слушаю HTTP трафик (порт 80)...")
#sniff(filter="tcp port 80", prn=process_http, store=0)
packets = sniff(filter="tcp port 80", prn=process_http, store=0, timeout=10)

print("Время вышло. Прослушивание завершено.")
print(f"Перехвачено пакетов: {len(packets)}")

#print(packets[0].summary())

if len(packets) > 0: # Сначала проверяем, поймали ли мы хоть что-то
    pkt = packets[0]  # Берем первый пакет из списка по индексу [0]
    
#pkt = packets[0]
#pkt.show()
#pkt = packets

if pkt.haslayer('Raw'):
#if packets.haslayer('Raw'):
    payload = pkt['Raw'].load
    print(f"Данные в байтах: {payload}")
    # Попытка декодировать в текст (если это не шифрованный трафик)
    try:
        print(f"Текст: {payload.decode('utf-8')}")
    except:
        print("Данные не являются текстом (бинарные или зашифрованы)")

Слушаю HTTP трафик (порт 80)...
Время вышло. Прослушивание завершено.
Перехвачено пакетов: 0


AttributeError: 'list' object has no attribute 'haslayer'

In [15]:
 print(f"Перехвачено пакетов: {len(packets)}")

Перехвачено пакетов: 0


In [24]:
from scapy.all import sniff

# 1. Собираем пакеты (например, 5 штук или по таймауту)
#packets = sniff(filter="tcp", count=5, timeout=10) 
#packets = sniff(filter="tcp", count=5, timeout=10) 
#packets = sniff(filter="tcp port 80", prn=process_http, store=0, timeout=10)
#packets = sniff(filter="tcp port 80", prn=process_http, timeout=10)
packets = sniff(filter="tcp port 80", count=5, prn=process_http, timeout=20)
               

print(f"Перехвачено пакетов: {len(packets)}")

# 2. Проверяем, что мы вообще что-то поймали
if len(packets) > 0:
    # ВАЖНО: берем именно первый элемент через [0]
    pkt = packets[0] 
    
    # Теперь pkt — это ОДИН пакет, у него ЕСТЬ метод haslayer
    if pkt.haslayer('Raw'):
        payload = pkt['Raw'].load
        print(f"Данные первого пакета: {payload}")
    else:
        print("В первом пакете нет слоя Raw (полезной нагрузки)")
else:
    print("Пакеты не были перехвачены.")


Перехвачено пакетов: 2
Данные первого пакета: b'\x00'


In [29]:
pkt[0].show()
if pkt.haslayer('IP'):
    src_ip = pkt['IP'].src
    dst_ip = pkt['IP'].dst
    version_ip = pkt['IP'].version
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")
    print(f"Версия IP: {version_ip}")

###[ Ethernet ]###
  dst       = 40:3f:8c:e0:94:5e
  src       = 00:d8:61:a5:da:5e
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 41
     id        = 9418
     flags     = DF
     frag      = 0
     ttl       = 128
     proto     = tcp
     chksum    = 0x0
     src       = 192.168.0.101
     dst       = 173.194.222.102
     \options   \
###[ TCP ]###
        sport     = 60231
        dport     = http
        seq       = 3257996799
        ack       = 3723813007
        dataofs   = 5
        reserved  = 0
        flags     = A
        window    = 512
        chksum    = 0x4d52
        urgptr    = 0
        options   = []
###[ HTTP 1 ]###
###[ Raw ]###
           load      = b'\x00'

Откуда: 192.168.0.101 -> Куда: 173.194.222.102
Версия IP: 4


In [35]:
pkt[2].show()
if pkt.haslayer('IP'):
    src_ip = pkt['IP'].src
    dst_ip = pkt['IP'].dst
    version_ip = pkt['IP'].version
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")
    print(f"Версия IP: {version_ip}")

###[ TCP ]###
  sport     = 60231
  dport     = http
  seq       = 3257996799
  ack       = 3723813007
  dataofs   = 5
  reserved  = 0
  flags     = A
  window    = 512
  chksum    = 0x4d52
  urgptr    = 0
  options   = []
###[ HTTP 1 ]###
###[ Raw ]###
     load      = b'\x00'

Откуда: 192.168.0.101 -> Куда: 173.194.222.102
Версия IP: 4


In [36]:
from scapy.all import sniff, IP, TCP, Raw

# 1. Определяем ваш локальный IP (подставьте свой, если знаете)
# Или скрипт сам поймет, что пакет идет ОТ вас, по направлению к порту 80
def process_outgoing_http(packet):
    # Проверяем: есть ли IP, TCP и данные (Raw)
    if packet.haslayer(IP) and packet.haslayer(Raw):
        # Фильтруем: исходящий порт сервера должен быть 80 (HTTP)
        # А направление: от нас (локальный IP) к ним
        if packet[TCP].dport == 80:
            payload = packet[Raw].load.decode('utf-8', errors='ignore')
            
            # Ищем начало HTTP-запроса (GET, POST, PUT и т.д.)
            if any(method in payload for method in ["GET", "POST", "HTTP/1.1"]):
                print(f"\n[=>] ИСХОДЯЩИЙ ЗАПРОС на {packet[IP].dst}:")
                print("-" * 50)
                # Выводим первые несколько строк (заголовки)
                print("\n".join(payload.splitlines()[:5])) 
                print("-" * 50)

# 2. Запуск сниффера
# lfilter позволяет дополнительно отсеять входящие пакеты (на лету)
print("Отслеживаю исходящий HTTP-трафик...")
#sniff(filter="tcp port 80", prn=process_outgoing_http, store=0)
sniff(filter="tcp port 80", prn=process_outgoing_http, count=5)


Отслеживаю исходящий HTTP-трафик...


<Sniffed: TCP:5 UDP:0 ICMP:0 Other:0>

In [38]:
pkt = packet[0]
pkt.show()
if pkt.haslayer('IP'):
    src_ip = pkt['IP'].src
    dst_ip = pkt['IP'].dst
    version_ip = pkt['IP'].version
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")
    print(f"Версия IP: {version_ip}")

###[ Ethernet ]###
  dst       = 40:3f:8c:e0:94:5e
  src       = 00:d8:61:a5:da:5e
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 557
     id        = 11670
     flags     = DF
     frag      = 0
     ttl       = 128
     proto     = tcp
     chksum    = 0x0
     src       = 192.168.0.101
     dst       = 209.85.233.138
     \options   \
###[ TCP ]###
        sport     = 60239
        dport     = https
        seq       = 1842861127
        ack       = 2091798138
        dataofs   = 5
        reserved  = 0
        flags     = PA
        window    = 512
        chksum    = 0x7e0d
        urgptr    = 0
        options   = []
###[ Raw ]###
           load      = b'\x16\x03\x01\x02\x00\x01\x00\x01\xfc\x03\x03k\x9e\xbb|\x9e\xb0+\x87\xbf\x90L\xfe\x9cH[M%\x86\xa2\xc8f8\xc9\x83&\xfa\x18\xbd}\x92U6 \x1cr\x88\xb8K\xb5\x8a\xa5\xa4&KE\xd6\x13\x94\x8bQ\x817\xea0Pi\xbdO\x89\x9c\xdbO\x8b\xcc\x91\x00\x1e\x13\x01\x13\x03\x13\x02\xc0+\xc0/\xcc

In [2]:
from scapy.all import sniff
# Захватить 10 пакетов и вывести их краткое содержание
packets = sniff(count=10)
packets.summary()

Ether / IP / TCP 192.168.0.101:61372 > 208.115.231.38:https A / Raw
Ether / IP / TCP 208.115.231.38:https > 192.168.0.101:61372 A / Padding
Ether / IP / TCP 192.168.0.101:61372 > 208.115.231.38:https A
Ether / IP / TCP 208.115.231.38:https > 192.168.0.101:61372 A
Ether / IP / TCP 192.168.0.101:63417 > 149.154.167.99:https PA / Raw
Ether / IP / TCP 192.168.0.101:63416 > 149.154.167.99:https PA / Raw
Ether / IP / TCP 192.168.0.101:56720 > 149.154.167.99:https PA / Raw
Ether / IP / TCP 192.168.0.101:63418 > 149.154.167.99:https PA / Raw
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:56720 A / Padding
Ether / IP / TCP 149.154.167.99:https > 192.168.0.101:63417 PA / Raw


In [3]:
pakets = sniff(filter="tcp and port 80", prn=lambda x: x.summary(), count=10)

Ether / IP / TCP 192.168.0.101:52864 > 108.177.14.139:http A / Raw
Ether / IP / TCP 108.177.14.139:http > 192.168.0.101:52864 A
Ether / IP / TCP 192.168.0.101:53481 > 8.6.112.6:http S
Ether / IP / TCP 8.6.112.6:http > 192.168.0.101:53481 SA
Ether / IP / TCP 192.168.0.101:53481 > 8.6.112.6:http A
Ether / IP / TCP 192.168.0.101:53481 > 8.6.112.6:http PA / Raw
Ether / IP / TCP 8.6.112.6:http > 192.168.0.101:53481 A / Padding
Ether / IP / TCP 8.6.112.6:http > 192.168.0.101:53481 A / Raw
Ether / IP / TCP 8.6.112.6:http > 192.168.0.101:53481 A / Raw
Ether / IP / TCP 192.168.0.101:53481 > 8.6.112.6:http A


In [7]:
n = 3
pakets[n].show()
if pakets[n].haslayer('IP'):
    src_ip = pakets[n]['IP'].src
    dst_ip = pakets[n]['IP'].dst
    version_ip = pakets[n]['IP'].version
    
    #agent = pakets[n]['HTTP Request'].User_Agent.decode()
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")
    print(f"Версия IP: {version_ip}")
    #print(f"User-Agent: {agent}")

###[ Ethernet ]###
  dst       = 00:d8:61:a5:da:5e
  src       = 40:3f:8c:e0:94:5e
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 52
     id        = 0
     flags     = DF
     frag      = 0
     ttl       = 55
     proto     = tcp
     chksum    = 0xaab
     src       = 8.6.112.6
     dst       = 192.168.0.101
     \options   \
###[ TCP ]###
        sport     = http
        dport     = 53481
        seq       = 1522865612
        ack       = 2466674854
        dataofs   = 8
        reserved  = 0
        flags     = SA
        window    = 65535
        chksum    = 0xe869
        urgptr    = 0
        options   = [('MSS', 1460), ('NOP', None), ('NOP', None), ('SAckOK', b''), ('NOP', None), ('WScale', 13)]

Откуда: 8.6.112.6 -> Куда: 192.168.0.101
Версия IP: 4


In [55]:
pakets = sniff(filter="tcp and port 80", prn=lambda x: x.summary(), count=10)

Ether / IP / TCP / HTTP / GET '/generate_204' 
Ether / IP / TCP 108.177.14.100:http > 192.168.0.101:49935 A / Padding
Ether / IP / TCP / HTTP / 204 No Content
Ether / IP / TCP 192.168.0.101:49935 > 108.177.14.100:http A
Ether / IP / TCP 192.168.0.101:49935 > 108.177.14.100:http A / HTTP / Raw
Ether / IP / TCP 108.177.14.100:http > 192.168.0.101:49935 A
Ether / IP / TCP / HTTP / GET '/generate_204' 
Ether / IP / TCP 192.168.0.101:49935 > 108.177.14.100:http A / HTTP / Raw
Ether / IP / TCP 108.177.14.100:http > 192.168.0.101:49935 A / Padding
Ether / IP / TCP / HTTP / 204 No Content


In [56]:
n = 6
pakets[n].show()
if pakets[n].haslayer('IP'):
    src_ip = pakets[n]['IP'].src
    dst_ip = pakets[n]['IP'].dst
    version_ip = pakets[n]['IP'].version
    #agent = 
    agent = pakets[n]['HTTP Request'].User_Agent.decode()
    print(f"Откуда: {src_ip} -> Куда: {dst_ip}")
    print(f"Версия IP: {version_ip}")
    print(f"User-Agent: {agent}")

###[ Ethernet ]###
  dst       = 40:3f:8c:e0:94:5e
  src       = 00:d8:61:a5:da:5e
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 152
     id        = 14166
     flags     = DF
     frag      = 0
     ttl       = 128
     proto     = tcp
     chksum    = 0x0
     src       = 192.168.0.101
     dst       = 108.177.14.100
     \options   \
###[ TCP ]###
        sport     = 49935
        dport     = http
        seq       = 2426658356
        ack       = 2791782660
        dataofs   = 5
        reserved  = 0
        flags     = PA
        window    = 511
        chksum    = 0x3cad
        urgptr    = 0
        options   = []
###[ HTTP 1 ]###
###[ HTTP Request ]###
           Method    = GET
           Path      = /generate_204
           Http_Version= HTTP/1.1
           A_IM      = None
           Accept    = None
           Accept_Charset= None
           Accept_Datetime= None
           Accept_Encoding= gzip
           Accept

In [8]:
print("Получилось перехватить HTTP трафик, но что дальше делать и как совершенно не понятно...")

Получилось перехватить HTTP трафик, но что дальше делать и как совершенно не понятно...
